# Hyperparmeter optimizati nusing Scikit-learn and optuna

In [1]:
import rdkit
from rdkit import Chem
from rdkit.Chem import AllChem
from tqdm import tqdm

- drugs.csv 파일 안에는 실제 FDA의 승인을 받은 약과 약물이 아닌 non-drug 들이 존재한다

In [2]:
import pandas as pd
import numpy as np

## 1. 데이터 읽어들이기

In [5]:
drugs = pd.read_csv("data/drugs_7강.csv")

In [6]:
drugs

,smiles,is_drug
0,BrC1=CC2=C(NC(=O)CN=C2C2=CC=CC=N2)C=C1,1
1,C#CCN[C@@H]1CCC2=CC=CC=C12,1
2,C1(CC[C@@]2([C@@H](CC(N)=O)[C@@]3([C@@]4([N+]5...,1
3,C1C2CNCC1C1=C2C=C2N=CC=NC2=C1,1
4,C1CN2C[C@@H](N=C2S1)C1=CC=CC=C1,1
...,...,...
595,CO[C@H]1[C@H](O)CC(=O)O[C@H](C)C\C=C\C=C\[C@H]...,1
596,CO[C@H]1\C=C\O[C@@]2(C)OC3=C(C)C(O)=C4C(O)=C(N...,1
597,CO[C@H]1\C=C\O[C@@]2(C)OC3=C(C)C(O)=C4C(O)=C(N...,1
598,CO[C@H]1\C=C\O[C@@]2(C)OC3=C(C2=O)C2=C(C(O)=C3...,1


* 실습 목적: Drug과 non-drug 구분하는 모델 만들기

* 그 후, 이 모델의 성능을 최적화 시키는 연습하기

# 실습

## 1. Descriptor로 변환

In [7]:
from rdkit.Chem import AllChem
from rdkit import DataStructs
from rdkit.Chem import MolFromSmiles
from rdkit.Chem.GraphDescriptors import (BalabanJ, BertzCT, Chi0, Chi0n, Chi0v, Chi1,
                                         Chi1n, Chi1v, Chi2n, Chi2v, Chi3n, Chi3v, Chi4n, Chi4v,
                                         HallKierAlpha, Ipc, Kappa1, Kappa2, Kappa3)

from rdkit.Chem.EState.EState_VSA import (EState_VSA1, EState_VSA10, EState_VSA11, EState_VSA2, EState_VSA3,
                                          EState_VSA4, EState_VSA5, EState_VSA6, EState_VSA7, EState_VSA8, EState_VSA9,
                                          VSA_EState1, VSA_EState10, VSA_EState2, VSA_EState3, VSA_EState4, VSA_EState5,
                                          VSA_EState6, VSA_EState7, VSA_EState8, VSA_EState9,)

from rdkit.Chem.Descriptors import (ExactMolWt, MolWt, HeavyAtomMolWt, MaxAbsPartialCharge, MinPartialCharge,
                                    MaxPartialCharge, MinAbsPartialCharge, NumRadicalElectrons, NumValenceElectrons)

from rdkit.Chem.EState.EState import (MaxAbsEStateIndex, MaxEStateIndex, MinAbsEStateIndex, MinEStateIndex,)

from rdkit.Chem.Lipinski import (FractionCSP3, HeavyAtomCount, NHOHCount, NOCount, NumAliphaticCarbocycles,
                                 NumAliphaticHeterocycles, NumAliphaticRings, NumAromaticCarbocycles, NumAromaticHeterocycles,
                                 NumAromaticRings, NumHAcceptors, NumHDonors, NumHeteroatoms, RingCount,
                                 NumRotatableBonds, NumSaturatedCarbocycles, NumSaturatedHeterocycles, NumSaturatedRings,)

from rdkit.Chem.Crippen import (MolLogP, MolMR, )

from rdkit.Chem.MolSurf import (LabuteASA, PEOE_VSA1, PEOE_VSA10, PEOE_VSA11, PEOE_VSA12, PEOE_VSA13, PEOE_VSA14,
                                PEOE_VSA2, PEOE_VSA3,PEOE_VSA4, PEOE_VSA5, PEOE_VSA6, PEOE_VSA7, PEOE_VSA8, PEOE_VSA9,
                                SMR_VSA1, SMR_VSA10, SMR_VSA2, SMR_VSA3, SMR_VSA4, SMR_VSA5, SMR_VSA6,
                                SMR_VSA7, SMR_VSA8, SMR_VSA9, SlogP_VSA1, SlogP_VSA10, SlogP_VSA11, SlogP_VSA12,
                                SlogP_VSA2, SlogP_VSA3,SlogP_VSA4, SlogP_VSA5, SlogP_VSA6, SlogP_VSA7, SlogP_VSA8,
                                SlogP_VSA9, TPSA, )

from rdkit.Chem.Fragments import (fr_Al_COO, fr_Al_OH, fr_Al_OH_noTert, fr_ArN, fr_Ar_COO, fr_Ar_N, fr_Ar_NH,
 fr_Ar_OH, fr_COO, fr_COO2, fr_C_O, fr_C_O_noCOO, fr_C_S, fr_HOCCN, fr_Imine, fr_NH0, fr_NH1,
 fr_NH2, fr_N_O, fr_Ndealkylation1, fr_Ndealkylation2, fr_Nhpyrrole, fr_SH, fr_aldehyde, fr_alkyl_carbamate,
 fr_alkyl_halide, fr_allylic_oxid, fr_amide, fr_amidine, fr_aniline, fr_aryl_methyl, fr_azide, fr_azo, fr_barbitur,
 fr_benzene, fr_benzodiazepine, fr_bicyclic, fr_diazo, fr_dihydropyridine, fr_epoxide, fr_ester, fr_ether, fr_furan,
 fr_guanido, fr_halogen, fr_hdrzine, fr_hdrzone, fr_imidazole, fr_imide, fr_isocyan, fr_isothiocyan, fr_ketone,
 fr_ketone_Topliss, fr_lactam, fr_lactone, fr_methoxy, fr_morpholine, fr_nitrile, fr_nitro, fr_nitro_arom,
 fr_nitro_arom_nonortho, fr_nitroso, fr_oxazole, fr_oxime, fr_para_hydroxylation, fr_phenol,
 fr_phenol_noOrthoHbond, fr_phos_acid, fr_phos_ester, fr_piperdine, fr_piperzine, fr_priamide, fr_prisulfonamd,
 fr_pyridine, fr_quatN, fr_sulfide, fr_sulfonamd, fr_sulfone, fr_term_acetylene, fr_tetrazole, fr_thiazole, fr_thiocyan,
 fr_thiophene, fr_unbrch_alkane, fr_urea)

# Descriptor 계산 수행 함수. 
def calc_descriptors(mol):
    if mol is None:
        print("Molecule is None!")
        return None
    else:
        AllChem.ComputeGasteigerCharges(mol)
        finger = [
            BalabanJ(mol) , # 0
            BertzCT(mol) , # 1
            Chi0(mol) , # 2
            Chi0n(mol) , # 3
            Chi0v(mol) , # 4
            Chi1(mol) , # 5
            Chi1n(mol) , # 6
            Chi1v(mol) , # 7
            Chi2n(mol) ,
            Chi2v(mol) ,
            Chi3n(mol) ,
            Chi3v(mol) ,
            Chi4n(mol) ,
            Chi4v(mol) ,
            EState_VSA1(mol) ,
            EState_VSA10(mol) ,
            EState_VSA11(mol) ,
            EState_VSA2(mol) ,
            EState_VSA3(mol) ,
            EState_VSA4(mol) ,
            EState_VSA5(mol) ,
            EState_VSA6(mol) ,
            EState_VSA7(mol) ,
            EState_VSA8(mol) ,
                EState_VSA9(mol) ,
                ExactMolWt(mol) ,
                FractionCSP3(mol) ,
                HallKierAlpha(mol) ,
                HeavyAtomCount(mol) ,
                HeavyAtomMolWt(mol) ,
                # Ipc(mol) ,
                Kappa1(mol) ,
                Kappa2(mol) ,
                Kappa3(mol) ,
                LabuteASA(mol) ,
                MaxAbsEStateIndex(mol) ,
                MaxAbsPartialCharge(mol) ,
                MaxEStateIndex(mol) ,
                MaxPartialCharge(mol) ,
                MinAbsEStateIndex(mol) ,
                MinAbsPartialCharge(mol) ,
                MinEStateIndex(mol) ,
                MinPartialCharge(mol) ,
                MolLogP(mol) ,
                MolMR(mol) ,
                MolWt(mol) ,
                NHOHCount(mol) ,
                NOCount(mol) ,
                NumAliphaticCarbocycles(mol) ,
                NumAliphaticHeterocycles(mol) ,
                NumAliphaticRings(mol) ,
                NumAromaticCarbocycles(mol) ,
                NumAromaticHeterocycles(mol) ,
                NumAromaticRings(mol) ,
                NumHAcceptors(mol) ,
                NumHDonors(mol) ,
                NumHeteroatoms(mol) ,
                NumRadicalElectrons(mol) ,
                NumRotatableBonds(mol) ,
                NumSaturatedCarbocycles(mol) ,
                NumSaturatedHeterocycles(mol) ,
                NumSaturatedRings(mol) ,
                NumValenceElectrons(mol) ,
                PEOE_VSA1(mol) ,
                PEOE_VSA10(mol) ,
                PEOE_VSA11(mol) ,
                PEOE_VSA12(mol) ,
                PEOE_VSA13(mol) ,
                PEOE_VSA14(mol) ,
                PEOE_VSA2(mol) ,
                PEOE_VSA3(mol) ,
                PEOE_VSA4(mol) ,
                PEOE_VSA5(mol) ,
                PEOE_VSA6(mol) ,
                PEOE_VSA7(mol) ,
                PEOE_VSA8(mol) ,
                PEOE_VSA9(mol) ,
                RingCount(mol) ,
                SMR_VSA1(mol) ,
                SMR_VSA10(mol) ,
                SMR_VSA2(mol) ,
                SMR_VSA3(mol) ,
                SMR_VSA4(mol) ,
                SMR_VSA5(mol) ,
                SMR_VSA6(mol) ,
                SMR_VSA7(mol) ,
                SMR_VSA8(mol) ,
                SMR_VSA9(mol) ,
                SlogP_VSA1(mol) ,
                SlogP_VSA10(mol) ,
                SlogP_VSA11(mol) ,
                SlogP_VSA12(mol) ,
                SlogP_VSA2(mol) ,
                SlogP_VSA3(mol) ,
                SlogP_VSA4(mol) ,
                SlogP_VSA5(mol) ,
                SlogP_VSA6(mol) ,
                SlogP_VSA7(mol) ,
                SlogP_VSA8(mol) ,
                SlogP_VSA9(mol) ,
                TPSA(mol) ,
                VSA_EState1(mol) ,
                VSA_EState10(mol) ,
                VSA_EState2(mol) ,
                VSA_EState3(mol) ,
                VSA_EState4(mol) ,
                VSA_EState5(mol) ,
                VSA_EState6(mol) ,
                VSA_EState7(mol) ,
                VSA_EState8(mol) ,
                VSA_EState9(mol) ,
                fr_Al_COO(mol) ,
                fr_Al_OH(mol) ,
                fr_Al_OH_noTert(mol) ,
                fr_ArN(mol) ,
                fr_Ar_COO(mol) ,
                fr_Ar_N(mol) ,
                fr_Ar_NH(mol) ,
                fr_Ar_OH(mol) ,
                fr_COO(mol) ,
                fr_COO2(mol) ,
                fr_C_O(mol) ,
                fr_C_O_noCOO(mol) ,
                fr_C_S(mol) ,
                fr_HOCCN(mol) ,
                fr_Imine(mol) ,
                fr_NH0(mol) ,
                fr_NH1(mol) ,
                fr_NH2(mol) ,
                fr_N_O(mol) ,
                fr_Ndealkylation1(mol) ,
                fr_Ndealkylation2(mol) ,
                fr_Nhpyrrole(mol) ,
                fr_SH(mol) ,
                fr_aldehyde(mol) ,
                fr_alkyl_carbamate(mol) ,
                fr_alkyl_halide(mol) ,
                fr_allylic_oxid(mol) ,
                fr_amide(mol) ,
                fr_amidine(mol) ,
                fr_aniline(mol) ,
                fr_aryl_methyl(mol) ,
                fr_azide(mol) ,
                fr_azo(mol) ,
                fr_barbitur(mol) ,
                fr_benzene(mol) ,
                fr_benzodiazepine(mol) ,
                fr_bicyclic(mol) ,
                fr_diazo(mol) ,
                fr_dihydropyridine(mol) ,
                fr_epoxide(mol) ,
                fr_ester(mol) ,
                fr_ether(mol) ,
                fr_furan(mol) ,
                fr_guanido(mol) ,
                fr_halogen(mol) ,
                fr_hdrzine(mol) ,
                fr_hdrzone(mol) ,
                fr_imidazole(mol) ,
                fr_imide(mol) ,
                fr_isocyan(mol) ,
                fr_isothiocyan(mol) ,
                fr_ketone(mol) ,
                fr_ketone_Topliss(mol) ,
                fr_lactam(mol) ,
                fr_lactone(mol) ,
                fr_methoxy(mol) ,
                fr_morpholine(mol) ,
                fr_nitrile(mol) ,
                fr_nitro(mol) ,
                fr_nitro_arom(mol) ,
                fr_nitro_arom_nonortho(mol) ,
                fr_nitroso(mol) ,
                fr_oxazole(mol) ,
                fr_oxime(mol) ,
                fr_para_hydroxylation(mol) ,
                fr_phenol(mol) ,
                fr_phenol_noOrthoHbond(mol) ,
                fr_phos_acid(mol) ,
                fr_phos_ester(mol) ,
                fr_piperdine(mol) ,
                fr_piperzine(mol) ,
                fr_priamide(mol) ,
                fr_prisulfonamd(mol) ,
                fr_pyridine(mol) ,
                fr_quatN(mol) ,
                fr_sulfide(mol) ,
                fr_sulfonamd(mol) ,
                fr_sulfone(mol) ,
                fr_term_acetylene(mol) ,
                fr_tetrazole(mol) ,
                fr_thiazole(mol) ,
                fr_thiocyan(mol) ,
                fr_thiophene(mol),
                fr_unbrch_alkane(mol) ,
                fr_urea(mol) , #rdkit properties # 196
                ]
        return finger

In [13]:
import math   # math 함수에는 'is None' 이라는 함수가 있다.

desc_list = []   # descriptor 리스트
is_drug = []  # True/False 저장할 리스트
for smi, flag in zip(drugs["smiles"], drugs["is_drug"]):
    m = Chem.MolFromSmiles(smi)
    if m is None:   # smiles에서 mol로 변환할때, 에러가 나면 즉, smiles가 문제가 있을 때, skip!
        continue
    desc = calc_descriptors(m)

    ## 아래 any 문은 실행안함.
    # NaN 존재하는지 확인 후, NaN이 있으면 skip!
    if any([math.isnan(v) for v in desc]):   # desc 리스트에서 v를 하나씩 뽑아올때, is NaN이 있다면 skip 
        print(f"{smi} cause NaN")
        continue
        
    desc_list.append(desc)
    if flag == 1:
        is_drug.append(True)
    else:
        is_drug.append(False)

++ 추가 정보 ++
1. zip
   
  : 여러 개의 리스트를 하나로 묶어주는 함수


  : drugs["smiles"] 리스트와 drugs["is_drug"] 리스트의 첫 번째 값을 한 쌍으로 묶고 두 번째 값들끼리 묶는다.

2. flag
  : 프로그래밍에서 상태를 나타내는 변수명으로 자주 쓰임


  : 즉, zip으로 묶어주는 리스트 개수와 for 문의 변수는 동일해야한다.
  
  : drugs["smiles"] = 변수 smi에 할당되고, drug["is_drug"] = 변수 flag에 할당된다.

3. math. isnan(v) for v in desc
   
  : desc 안에 있는 값들을 하나씩 꺼내서 확인

  : math.isnan(v): 이 값(v)이 NaN(계산 불능/빈 값) 이니? 라고 묻는 함수
  => 결과적으로 NaN인 항목은 True로 표시된 리스트가 만들어짐
  
4. if any()
   : any() 리스트 안에 True가 단 하나라도 있으면 전체를 True로 판단

In [14]:
len(desc_list)

600

In [15]:
len(is_drug)

600

In [17]:
X = np.array(desc_list)

In [18]:
is_drug

[True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,

### Troubleshooting

* is_dug 출력시, 모두 True만 출력
* 망할 drugs에는 1만 있고 non_drugs에 0만 있는 파일임... Troubleshooting 할게 없음 맞게 나온거니까...;;(내 시간...)

In [20]:
print(drugs["is_drug"].value_counts())

is_drug
1    600
Name: count, dtype: int64


In [21]:
set(is_drug)

{True}

-----------------------------------------------------

# Hyperparmeter optimizati nusing Scikit-learn and optuna

In [22]:
import rdkit
from rdkit import Chem
from rdkit.Chem import AllChem
from tqdm import tqdm

* drugs.csv 파일 안에는 실제 FDA의 승인을 받은 약과 약물이 아닌 non-drug 들이 존재한다

In [23]:
import pandas as pd
import numpy as np

## 1. 데이터 읽어들이기

In [26]:
# 각각ㅢ 팡리 불러오기
drugs = pd.read_csv("data/drugs_7강.csv")
non_drugs = pd.read_csv("data/non_drug_7강.csv")

In [29]:
# 두 데이터 브레임을 하나로 합치기 (위 아래로 연결)
# axis=0은 행 방향(위 아래)으로 합친다는 의미
combined_drugs = pd.concat([drugs, non_drugs], axis=0)

In [30]:
# 인덱스 초기화 (합치고 나면 번호가 꼬일 수 있으므로 새로 부여)
combined_drugs = combined_drugs.reset_index(drop=True)

In [33]:
from rdkit.Chem import AllChem
from rdkit import DataStructs
from rdkit.Chem import MolFromSmiles
from rdkit.Chem.GraphDescriptors import (BalabanJ, BertzCT, Chi0, Chi0n, Chi0v, Chi1,
                                         Chi1n, Chi1v, Chi2n, Chi2v, Chi3n, Chi3v, Chi4n, Chi4v,
                                         HallKierAlpha, Ipc, Kappa1, Kappa2, Kappa3)

from rdkit.Chem.EState.EState_VSA import (EState_VSA1, EState_VSA10, EState_VSA11, EState_VSA2, EState_VSA3,
                                          EState_VSA4, EState_VSA5, EState_VSA6, EState_VSA7, EState_VSA8, EState_VSA9,
                                          VSA_EState1, VSA_EState10, VSA_EState2, VSA_EState3, VSA_EState4, VSA_EState5,
                                          VSA_EState6, VSA_EState7, VSA_EState8, VSA_EState9,)

from rdkit.Chem.Descriptors import (ExactMolWt, MolWt, HeavyAtomMolWt, MaxAbsPartialCharge, MinPartialCharge,
                                    MaxPartialCharge, MinAbsPartialCharge, NumRadicalElectrons, NumValenceElectrons)

from rdkit.Chem.EState.EState import (MaxAbsEStateIndex, MaxEStateIndex, MinAbsEStateIndex, MinEStateIndex,)

from rdkit.Chem.Lipinski import (FractionCSP3, HeavyAtomCount, NHOHCount, NOCount, NumAliphaticCarbocycles,
                                 NumAliphaticHeterocycles, NumAliphaticRings, NumAromaticCarbocycles, NumAromaticHeterocycles,
                                 NumAromaticRings, NumHAcceptors, NumHDonors, NumHeteroatoms, RingCount,
                                 NumRotatableBonds, NumSaturatedCarbocycles, NumSaturatedHeterocycles, NumSaturatedRings,)

from rdkit.Chem.Crippen import (MolLogP, MolMR, )

from rdkit.Chem.MolSurf import (LabuteASA, PEOE_VSA1, PEOE_VSA10, PEOE_VSA11, PEOE_VSA12, PEOE_VSA13, PEOE_VSA14,
                                PEOE_VSA2, PEOE_VSA3,PEOE_VSA4, PEOE_VSA5, PEOE_VSA6, PEOE_VSA7, PEOE_VSA8, PEOE_VSA9,
                                SMR_VSA1, SMR_VSA10, SMR_VSA2, SMR_VSA3, SMR_VSA4, SMR_VSA5, SMR_VSA6,
                                SMR_VSA7, SMR_VSA8, SMR_VSA9, SlogP_VSA1, SlogP_VSA10, SlogP_VSA11, SlogP_VSA12,
                                SlogP_VSA2, SlogP_VSA3,SlogP_VSA4, SlogP_VSA5, SlogP_VSA6, SlogP_VSA7, SlogP_VSA8,
                                SlogP_VSA9, TPSA, )

from rdkit.Chem.Fragments import (fr_Al_COO, fr_Al_OH, fr_Al_OH_noTert, fr_ArN, fr_Ar_COO, fr_Ar_N, fr_Ar_NH,
 fr_Ar_OH, fr_COO, fr_COO2, fr_C_O, fr_C_O_noCOO, fr_C_S, fr_HOCCN, fr_Imine, fr_NH0, fr_NH1,
 fr_NH2, fr_N_O, fr_Ndealkylation1, fr_Ndealkylation2, fr_Nhpyrrole, fr_SH, fr_aldehyde, fr_alkyl_carbamate,
 fr_alkyl_halide, fr_allylic_oxid, fr_amide, fr_amidine, fr_aniline, fr_aryl_methyl, fr_azide, fr_azo, fr_barbitur,
 fr_benzene, fr_benzodiazepine, fr_bicyclic, fr_diazo, fr_dihydropyridine, fr_epoxide, fr_ester, fr_ether, fr_furan,
 fr_guanido, fr_halogen, fr_hdrzine, fr_hdrzone, fr_imidazole, fr_imide, fr_isocyan, fr_isothiocyan, fr_ketone,
 fr_ketone_Topliss, fr_lactam, fr_lactone, fr_methoxy, fr_morpholine, fr_nitrile, fr_nitro, fr_nitro_arom,
 fr_nitro_arom_nonortho, fr_nitroso, fr_oxazole, fr_oxime, fr_para_hydroxylation, fr_phenol,
 fr_phenol_noOrthoHbond, fr_phos_acid, fr_phos_ester, fr_piperdine, fr_piperzine, fr_priamide, fr_prisulfonamd,
 fr_pyridine, fr_quatN, fr_sulfide, fr_sulfonamd, fr_sulfone, fr_term_acetylene, fr_tetrazole, fr_thiazole, fr_thiocyan,
 fr_thiophene, fr_unbrch_alkane, fr_urea)

# Descriptor 계산 수행 함수. 
def calc_descriptors(mol):
    if mol is None:
        print("Molecule is None!")
        return None
    else:
        AllChem.ComputeGasteigerCharges(mol)
        finger = [
            BalabanJ(mol) , # 0
            BertzCT(mol) , # 1
            Chi0(mol) , # 2
            Chi0n(mol) , # 3
            Chi0v(mol) , # 4
            Chi1(mol) , # 5
            Chi1n(mol) , # 6
            Chi1v(mol) , # 7
            Chi2n(mol) ,
            Chi2v(mol) ,
            Chi3n(mol) ,
            Chi3v(mol) ,
            Chi4n(mol) ,
            Chi4v(mol) ,
            EState_VSA1(mol) ,
            EState_VSA10(mol) ,
            EState_VSA11(mol) ,
            EState_VSA2(mol) ,
            EState_VSA3(mol) ,
            EState_VSA4(mol) ,
            EState_VSA5(mol) ,
            EState_VSA6(mol) ,
            EState_VSA7(mol) ,
            EState_VSA8(mol) ,
                EState_VSA9(mol) ,
                ExactMolWt(mol) ,
                FractionCSP3(mol) ,
                HallKierAlpha(mol) ,
                HeavyAtomCount(mol) ,
                HeavyAtomMolWt(mol) ,
                # Ipc(mol) ,
                Kappa1(mol) ,
                Kappa2(mol) ,
                Kappa3(mol) ,
                LabuteASA(mol) ,
                MaxAbsEStateIndex(mol) ,
                MaxAbsPartialCharge(mol) ,
                MaxEStateIndex(mol) ,
                MaxPartialCharge(mol) ,
                MinAbsEStateIndex(mol) ,
                MinAbsPartialCharge(mol) ,
                MinEStateIndex(mol) ,
                MinPartialCharge(mol) ,
                MolLogP(mol) ,
                MolMR(mol) ,
                MolWt(mol) ,
                NHOHCount(mol) ,
                NOCount(mol) ,
                NumAliphaticCarbocycles(mol) ,
                NumAliphaticHeterocycles(mol) ,
                NumAliphaticRings(mol) ,
                NumAromaticCarbocycles(mol) ,
                NumAromaticHeterocycles(mol) ,
                NumAromaticRings(mol) ,
                NumHAcceptors(mol) ,
                NumHDonors(mol) ,
                NumHeteroatoms(mol) ,
                NumRadicalElectrons(mol) ,
                NumRotatableBonds(mol) ,
                NumSaturatedCarbocycles(mol) ,
                NumSaturatedHeterocycles(mol) ,
                NumSaturatedRings(mol) ,
                NumValenceElectrons(mol) ,
                PEOE_VSA1(mol) ,
                PEOE_VSA10(mol) ,
                PEOE_VSA11(mol) ,
                PEOE_VSA12(mol) ,
                PEOE_VSA13(mol) ,
                PEOE_VSA14(mol) ,
                PEOE_VSA2(mol) ,
                PEOE_VSA3(mol) ,
                PEOE_VSA4(mol) ,
                PEOE_VSA5(mol) ,
                PEOE_VSA6(mol) ,
                PEOE_VSA7(mol) ,
                PEOE_VSA8(mol) ,
                PEOE_VSA9(mol) ,
                RingCount(mol) ,
                SMR_VSA1(mol) ,
                SMR_VSA10(mol) ,
                SMR_VSA2(mol) ,
                SMR_VSA3(mol) ,
                SMR_VSA4(mol) ,
                SMR_VSA5(mol) ,
                SMR_VSA6(mol) ,
                SMR_VSA7(mol) ,
                SMR_VSA8(mol) ,
                SMR_VSA9(mol) ,
                SlogP_VSA1(mol) ,
                SlogP_VSA10(mol) ,
                SlogP_VSA11(mol) ,
                SlogP_VSA12(mol) ,
                SlogP_VSA2(mol) ,
                SlogP_VSA3(mol) ,
                SlogP_VSA4(mol) ,
                SlogP_VSA5(mol) ,
                SlogP_VSA6(mol) ,
                SlogP_VSA7(mol) ,
                SlogP_VSA8(mol) ,
                SlogP_VSA9(mol) ,
                TPSA(mol) ,
                VSA_EState1(mol) ,
                VSA_EState10(mol) ,
                VSA_EState2(mol) ,
                VSA_EState3(mol) ,
                VSA_EState4(mol) ,
                VSA_EState5(mol) ,
                VSA_EState6(mol) ,
                VSA_EState7(mol) ,
                VSA_EState8(mol) ,
                VSA_EState9(mol) ,
                fr_Al_COO(mol) ,
                fr_Al_OH(mol) ,
                fr_Al_OH_noTert(mol) ,
                fr_ArN(mol) ,
                fr_Ar_COO(mol) ,
                fr_Ar_N(mol) ,
                fr_Ar_NH(mol) ,
                fr_Ar_OH(mol) ,
                fr_COO(mol) ,
                fr_COO2(mol) ,
                fr_C_O(mol) ,
                fr_C_O_noCOO(mol) ,
                fr_C_S(mol) ,
                fr_HOCCN(mol) ,
                fr_Imine(mol) ,
                fr_NH0(mol) ,
                fr_NH1(mol) ,
                fr_NH2(mol) ,
                fr_N_O(mol) ,
                fr_Ndealkylation1(mol) ,
                fr_Ndealkylation2(mol) ,
                fr_Nhpyrrole(mol) ,
                fr_SH(mol) ,
                fr_aldehyde(mol) ,
                fr_alkyl_carbamate(mol) ,
                fr_alkyl_halide(mol) ,
                fr_allylic_oxid(mol) ,
                fr_amide(mol) ,
                fr_amidine(mol) ,
                fr_aniline(mol) ,
                fr_aryl_methyl(mol) ,
                fr_azide(mol) ,
                fr_azo(mol) ,
                fr_barbitur(mol) ,
                fr_benzene(mol) ,
                fr_benzodiazepine(mol) ,
                fr_bicyclic(mol) ,
                fr_diazo(mol) ,
                fr_dihydropyridine(mol) ,
                fr_epoxide(mol) ,
                fr_ester(mol) ,
                fr_ether(mol) ,
                fr_furan(mol) ,
                fr_guanido(mol) ,
                fr_halogen(mol) ,
                fr_hdrzine(mol) ,
                fr_hdrzone(mol) ,
                fr_imidazole(mol) ,
                fr_imide(mol) ,
                fr_isocyan(mol) ,
                fr_isothiocyan(mol) ,
                fr_ketone(mol) ,
                fr_ketone_Topliss(mol) ,
                fr_lactam(mol) ,
                fr_lactone(mol) ,
                fr_methoxy(mol) ,
                fr_morpholine(mol) ,
                fr_nitrile(mol) ,
                fr_nitro(mol) ,
                fr_nitro_arom(mol) ,
                fr_nitro_arom_nonortho(mol) ,
                fr_nitroso(mol) ,
                fr_oxazole(mol) ,
                fr_oxime(mol) ,
                fr_para_hydroxylation(mol) ,
                fr_phenol(mol) ,
                fr_phenol_noOrthoHbond(mol) ,
                fr_phos_acid(mol) ,
                fr_phos_ester(mol) ,
                fr_piperdine(mol) ,
                fr_piperzine(mol) ,
                fr_priamide(mol) ,
                fr_prisulfonamd(mol) ,
                fr_pyridine(mol) ,
                fr_quatN(mol) ,
                fr_sulfide(mol) ,
                fr_sulfonamd(mol) ,
                fr_sulfone(mol) ,
                fr_term_acetylene(mol) ,
                fr_tetrazole(mol) ,
                fr_thiazole(mol) ,
                fr_thiocyan(mol) ,
                fr_thiophene(mol),
                fr_unbrch_alkane(mol) ,
                fr_urea(mol) , #rdkit properties # 196
                ]
        return finger

In [34]:
# 확인해보기
desc_list = []   # descriptor 리스트
is_drug = []  # True/False 저장할 리스트
for smi, flag in zip(combined_drugs["smiles"], combined_drugs["is_drug"]):
    m = Chem.MolFromSmiles(smi)
    if m is None:   # smiles에서 mol로 변환할때, 에러가 나면 즉, smiles가 문제가 있을 때, skip!
        continue
    desc = calc_descriptors(m)
    desc_list.append(desc)
    if flag == 1:
        is_drug.append(True)
    else:
        is_drug.append(False)

[14:31:43] WARNING: not removing hydrogen atom without neighbors
[14:32:17] WARNING: not removing hydrogen atom without neighbors
[14:32:17] WARNING: not removing hydrogen atom without neighbors
[14:32:56] WARNING: not removing hydrogen atom without neighbors
[14:33:05] WARNING: not removing hydrogen atom without neighbors
[14:33:33] WARNING: not removing hydrogen atom without neighbors
[14:33:33] WARNING: not removing hydrogen atom without neighbors
[14:33:33] WARNING: not removing hydrogen atom without neighbors
[14:33:33] WARNING: not removing hydrogen atom without neighbors
[14:33:33] WARNING: not removing hydrogen atom without neighbors


In [35]:
len(desc_list)
len(is_drug)

12290

In [36]:
len(desc_list)

12290

In [37]:
is_drug[:5]

[True, True, True, True, True]

In [38]:
is_drug[603:610]

[False, False, False, False, False, False, False]

## 2. training set, test set 만들기

In [39]:
X = np.array(desc_list)

In [40]:
y = np.array(is_drug)

In [41]:
from sklearn.model_selection import train_test_split

In [49]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state =42)

## 3. RandomForestClassifier 테스트

In [50]:
from sklearn.ensemble import RandomForestClassifier as RFC

In [51]:
my_model = RFC()

In [52]:
my_model.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [53]:
y_pred = my_model.predict(X_test)

## 4. 계산 tool

In [63]:
from sklearn.metrics import accuracy_score

In [55]:
accuracy_score(y_test, y_pred)

0.9491456468673718

In [59]:
# 어떤 parameter를 가지고 학습이 되었는지 확인 (get_params)
my_model.get_params()

{'bootstrap': True,
 'ccp_alpha': 0.0,
 'class_weight': None,
 'criterion': 'gini',
 'max_depth': None,
 'max_features': 'sqrt',
 'max_leaf_nodes': None,
 'max_samples': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'monotonic_cst': None,
 'n_estimators': 100,
 'n_jobs': None,
 'oob_score': False,
 'random_state': None,
 'verbose': 0,
 'warm_start': False}

## 5. Hyperparameter 찾는 도구
> GridSearchCV(격자 탐색): 정해진 범위 내의 모든 조합을 다 확인하는 방식으로 사용자가 미리 지정한 파라미터 후보들을 격자처럼만들어서, 모든 경우의 수를 다 대입
ex) 라면의 물 양(400, 500, 600ml)과 끓이는 시간 (3, 4, 5분)의 모든 조합(9가지)를 다 먹어보고 결정하는 것

> RandomizedSearch(랜덤 탐색): 모든 조합을 다 보지 않고, 정해진 횟수만큼만 무작위로 골라서 확인하는 방식으로 파라미터 범위를 설정해주면, 그 안에서 랜덤하게 값을 샘플링하여 성능 테스트
ex) 물 양과 시간을 수십 가지 경우의 수 중 무작위로 3가지만 뽑아서 요리해보고 그중 제일 나은 것을 고르는 것. 

In [60]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

In [64]:
params = {"n_estimators":[100, 150],  # n_estimators를 100과 150개를 test해보겠다.
          "min_samples_split": [2, 3, 4],    # min_samples_split의 parameter를  2, 3, 4로 test해보겠다.
          "max_features":[0.3, 0.5, 0.7]}    # max_features를 0.3, 0.5, 0.7로 test해보겠다.

* GridSearch를 수행한다는 것은 2 X 3 X 3의 18개의 parameter 조합을 테스트 하겠다.

* Default metrics 은 classification의 경우 => accuracy

* regression의 경우 => mean_squered_error

* metrics 은 다른 metrics을 사용 가능

### 5.1. GridSearchCV

In [65]:
clf = RFC(n_jobs = -1)     # n_jobs: 정수 값을 받는데, 몇개의 프로세스를 parrell 하게 쓸것인가? 'n_jobs = -1' =>  컴퓨터에 있는 모든 core를 사용하기때문에 빨라진다.
grid_search = GridSearchCV(clf, param_grid=params)

In [67]:
# ML과 마찬가지로 fit으로 학습시켜야 작동한다.
grid_search.fit(X, y)

,estimator,RandomForestC...ier(n_jobs=-1)
,param_grid,"{'max_features': [0.3, 0.5, ...], 'min_samples_split': [2, 3, ...], 'n_estimators': [100, 150]}"
,scoring,None
,n_jobs,None
,refit,True
,cv,None
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,100


In [69]:
# GridSearchCV와 Randomigrid_search.fit(X, y)zedSearchCV 가 끝난 결과는 cv_results_라는 attribute에 저장된다.
grid_search.cv_results_

{'mean_fit_time': array([15.43987274, 21.82034688, 13.85847621, 20.04867668, 13.74305453,
        20.57289042, 25.45077128, 35.79114046, 24.60146408, 36.01723399,
        29.07029104, 48.94019365, 54.46085262, 69.76484447, 45.42354317,
        83.89148159, 41.36716003, 56.68183112]),
 'std_fit_time': array([ 1.21046709,  1.15989089,  0.20318265,  0.2645185 ,  0.5065164 ,
         0.72975223,  1.26886442,  0.75550043,  1.36714737,  0.90239302,
         3.44413626,  3.01648992,  6.77655524, 10.46760967,  5.6295488 ,
        11.23154682,  3.0552836 ,  2.12163452]),
 'mean_score_time': array([0.07389107, 0.10076962, 0.07776546, 0.0985476 , 0.07124915,
        0.10440283, 0.06482911, 0.09709973, 0.0656363 , 0.09543023,
        0.09799438, 0.12521791, 0.12272706, 0.12449269, 0.09256077,
        0.21763253, 0.08975439, 0.11181707]),
 'std_score_time': array([0.01145286, 0.01134347, 0.01462674, 0.01564558, 0.01259496,
        0.01166144, 0.00594859, 0.00727546, 0.00598036, 0.00606984,
        

In [70]:
# 위의 cv_results_의 출력값은 읽기 어려웟 아래의 유틸리티를 이용한다.
# Utility function to report best scores
def report(results, n_top=3):
    for i in range(1, n_top + 1):
        candidates = np.flatnonzero(results['rank_test_score'] == i)
        for candidate in candidates:
            print("Model with rank: {0}".format(i))
            print("Mean validation score: {0:.3f} (std: {1:.3f})"
                  .format(results['mean_test_score'][candidate],
                          results['std_test_score'][candidate]))
            print("Parameters: {0}".format(results['params'][candidate]))
            print("")

In [71]:
report(grid_search.cv_results_)

Model with rank: 1
Mean validation score: 0.952 (std: 0.001)
Parameters: {'max_features': 0.3, 'min_samples_split': 2, 'n_estimators': 100}

Model with rank: 1
Mean validation score: 0.952 (std: 0.001)
Parameters: {'max_features': 0.3, 'min_samples_split': 4, 'n_estimators': 100}

Model with rank: 1
Mean validation score: 0.952 (std: 0.001)
Parameters: {'max_features': 0.3, 'min_samples_split': 4, 'n_estimators': 150}

Model with rank: 1
Mean validation score: 0.952 (std: 0.001)
Parameters: {'max_features': 0.7, 'min_samples_split': 3, 'n_estimators': 150}



* grid 를 사용하면 시간이 너무 오래걸림

  => Randomizedsearch를 이용 

### 5.2 RandomizedSearch

In [72]:
from time import time
n_iter = 5 # 랜덤하게 5번만 테스트. 
start = time()
random_search = RandomizedSearchCV(clf, param_distributions = params, n_iter = n_iter)
random_search.fit(X, y)
print("RandomizedSearchCV took %.2f seconds for %d candidates"
      " parameter settings." % ((time() - start), n_iter))   # 걸리는 시간 출력하는 함수

RandomizedSearchCV took 1061.86 seconds for 5 candidates parameter settings.


In [73]:
report(random_search.cv_results_)

Model with rank: 1
Mean validation score: 0.952 (std: 0.001)
Parameters: {'n_estimators': 150, 'min_samples_split': 4, 'max_features': 0.7}

Model with rank: 2
Mean validation score: 0.952 (std: 0.001)
Parameters: {'n_estimators': 150, 'min_samples_split': 2, 'max_features': 0.5}

Model with rank: 3
Mean validation score: 0.951 (std: 0.001)
Parameters: {'n_estimators': 100, 'min_samples_split': 2, 'max_features': 0.5}



# Optuna 라고 하는 hyperparameter package를 테스트

* https://optuna.org/

* 베이지안 최적화 방법을 통해서 단순 무작위가 아니라 기존의 결과에 기반하여 효율적으로 hyperparameter를 찾는다

* 목적함수 값(error)을 최소화하는 방향으로 모델 최적화를 수행함

* !pip install optuna 이 명령을 이용해서 설치 가능

In [75]:
!pip install optuna

## 1. 함수 정의

In [76]:
import optuna
import sklearn

def objective(trial): # trial은 optuna의 class이다. 
    
    # RandomForest 모델을 몇번 split할 것인가? 정수 값을 받는 파라미터. 
    # 정수로 정의되는 parameter는 suggest_int 라고 하는 method를 사용하여 범위 내 임의로 지정. 
    # 'rf_Max_depth'라는 parameter에서 2~32 중 아무 정수를 suggest를 하라
    rf_max_depth = trial.suggest_int('rf_max_depth', 2, 32)

    # n_estimator의 범위를 지정(50~200). 
    n_tree = trial.suggest_int('n_estimators', 50, 200)
    
    # min_samples_split이라고 하는 parameter는 2~5 사이에서 찾도록 지정. 
    min_samples_split = trial.suggest_int('min_samples_split', 2, 5)
    
    # max_features(전체 input feature 중에 몇 %를 node spilt로 사용할 것인지에 해당)는 실수로 주어지는 feature.
    # 그러므로 suggest_float 이라고하는 method를 사용. 
    max_features = trial.suggest_float('max_features', 0.3, 0.9)
    
    ## 여기까지 범위 지정 끝 ##
    
    # model을 정의
    regressor_obj = RFC(n_estimators = n_tree, 
                        min_samples_split = min_samples_split, 
                        max_depth=rf_max_depth, 
                        max_features = max_features, 
                        n_jobs=-1)
    # 모델 fitting. 
    regressor_obj.fit(X_train, y_train)

    # 모델을 validation set으로 검증. 
    y_pred = regressor_obj.predict(X_test)
    
    ## optuna에서는 목적함수 값을 최소화 하는 방향으로 최적화를 수행한다. 
    error = 1.0 - sklearn.metrics.accuracy_score(y_test, y_pred)
    
    return error

++ 추가 정보 ++

int => 정수
ex) suggest.int()

float => 실수
ex) suggest.float()

## 2. Study

In [77]:
study = optuna.create_study()
study.optimize(objective, n_trials=10)

[I 2026-03-28 17:10:21,575] A new study created in memory with name: no-name-c60acaf7-b519-4a33-9299-669ff76236e5
[I 2026-03-28 17:10:32,464] Trial 0 finished with value: 0.054922701383238404 and parameters: {'rf_max_depth': 5, 'n_estimators': 184, 'min_samples_split': 4, 'max_features': 0.5891955737910417}. Best is trial 0 with value: 0.054922701383238404.
[I 2026-03-28 17:10:59,372] Trial 1 finished with value: 0.05248169243287226 and parameters: {'rf_max_depth': 11, 'n_estimators': 187, 'min_samples_split': 5, 'max_features': 0.5305191592926195}. Best is trial 1 with value: 0.05248169243287226.
[I 2026-03-28 17:11:26,907] Trial 2 finished with value: 0.050854353132628205 and parameters: {'rf_max_depth': 26, 'n_estimators': 144, 'min_samples_split': 3, 'max_features': 0.40866668517873317}. Best is trial 2 with value: 0.050854353132628205.
[I 2026-03-28 17:11:37,240] Trial 3 finished with value: 0.052074857607811276 and parameters: {'rf_max_depth': 13, 'n_estimators': 90, 'min_samples

In [78]:
study.best_params

{'rf_max_depth': 26,
 'n_estimators': 144,
 'min_samples_split': 3,
 'max_features': 0.40866668517873317}

## 3. 예제 링크

* https://optuna.readthedocs.io/en/stable/tutorial/10_key_features/002_configurations.html#sphx-glr-tutorial-10-key-features-002-configurations-py